In [12]:
# Welcome to your new notebook
# Type here in the cell editor to add code!

df = spark.read.table("b_sales_transactions")

from pyspark.sql.functions import col, to_date, upper, trim, round, when, current_timestamp

df = df.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
df = df.withColumn("unit_price",   col("unit_price").cast("double"))
df = df.withColumn("quantity",     col("quantity").cast("integer"))
df = df.withColumn("discount_pct", col("discount_pct").cast("double"))
df = df.withColumn("shipping_cost",col("shipping_cost").cast("double"))

df.printSchema()

df = df.dropna(subset=["transaction_id", "order_date", "customer_id", "unit_price", "quantity"])
df = df.dropDuplicates(["transaction_id"])

print(f"Rows after cleaning: {df.count()}")

df = df.withColumn("customer_name", trim(col("customer_name")))
df = df.withColumn("city",          trim(col("city")))
df = df.withColumn("order_status",  trim(col("order_status")))
df = df.withColumn("category",      trim(col("category")))

df.show()

df = df.withColumn("gross_revenue", round(col("unit_price") * col("quantity"), 2))
df = df.withColumn("discount_amount", round(col("gross_revenue") * col("discount_pct") / 100, 2))
df = df.withColumn("net_revenue", round(col("gross_revenue") - col("discount_amount"), 2))
df = df.withColumn("is_returned", when(col("order_status") == "Returned", True).otherwise(False))

df.select("transaction_id", "unit_price", "quantity", "gross_revenue", "discount_amount", "net_revenue", "is_returned").show()

df.write.format("delta").mode("overwrite").saveAsTable("s_sales_transactions")
print("Silver table saved!")

StatementMeta(, 1de56a7d-7974-4331-a629-981404bb9f83, 95, Finished, Available, Finished, False)

root
 |-- transaction_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- region: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- sales_rep: string (nullable = true)

Rows after cleaning: 200
+--------------+----------+-----------+----------------+--------------------+----------------+------+-------+------